# Yahoo Finance (yfinance) 数据探针（单个 Ticker）

用途：**只观察 yfinance 能拉到什么数据、长什么样、缺什么**。  
不做估值计算。适合先用 `AAPL` / `MSFT` 跑通，再换你关心的标的。

> 说明：Yahoo Finance 的字段可能缺失或不稳定；本 notebook 会把缺失项明确显示出来。


In [ ]:
# 如果没有安装：pip install yfinance pandas
import pandas as pd
import yfinance as yf

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

## 1) 选择 Ticker 并拉取对象

In [ ]:
ticker = "AAPL"  # 改成你要看的，例如 "MSFT", "NVDA", "FTAI"
t = yf.Ticker(ticker)
ticker

## 2) 查看 `info`：哪些关键字段存在？

`info` 往往最不稳定，但对“市值、股本、倍数”很有用。下面只挑常用字段输出，并统计缺失情况。


In [ ]:
INFO_KEYS = [
    "shortName", "longName", "symbol", "currency",
    "sector", "industry", "exchange",
    "currentPrice", "previousClose",
    "marketCap", "enterpriseValue",
    "beta", "sharesOutstanding",
    "trailingPE", "forwardPE", "priceToBook",
    "trailingEps", "forwardEps",
    "ebitda", "totalRevenue",
    "grossMargins", "operatingMargins", "profitMargins",
]

def get_info(tkr: yf.Ticker) -> dict:
    try:
        return tkr.info or {}
    except Exception as e:
        print("Failed to read info:", e)
        return {}

info = get_info(t)
len(info), list(info.keys())[:20]

In [ ]:
rows = []
missing = []
for k in INFO_KEYS:
    v = info.get(k, None)
    rows.append((k, v))
    if v is None:
        missing.append(k)

info_df = pd.DataFrame(rows, columns=["field", "value"])
info_df

In [ ]:
# 缺失项
missing

In [ ]:
# 股本兜底估算：shares ≈ marketCap / currentPrice
mcap = info.get("marketCap")
px = info.get("currentPrice")
shares = info.get("sharesOutstanding")

shares_est = None
if shares is None and mcap and px:
    shares_est = mcap / px

pd.DataFrame([
    {"sharesOutstanding": shares, "sharesEst_mcap_div_px": shares_est}
])

## 3) 历史价格：看看 `history()` 返回结构

后续如果你要做“历史倍数分位/波动”，都要靠这一块。


In [ ]:
hist = t.history(period="1y", interval="1d", auto_adjust=False)
hist.tail()

## 4) 三张报表：年度与季度

yfinance 的报表 DataFrame 结构：
- 行（index）是科目（line items）
- 列（columns）是报告期（dates/periods）

我们会打印：shape、列名、科目前若干项、以及最近两期的截面。


In [ ]:
def summarize_stmt(df: pd.DataFrame, name: str, head_rows: int = 25):
    print(f"=== {name} ===")
    if df is None or df.empty:
        print("Empty / not available.\n")
        return
    print("shape:", df.shape)
    print("columns (periods) head:", list(df.columns)[:6], "..." if df.shape[1] > 6 else "")
    idx = list(df.index)
    print("rows (line items) head:", idx[:12], "..." if len(idx) > 12 else "")
    cols = list(df.columns)
    last_cols = cols[-2:] if len(cols) >= 2 else cols
    display(df[last_cols].head(head_rows))
    print()

# 年度
summarize_stmt(t.income_stmt, "income_stmt (annual)")
summarize_stmt(t.balance_sheet, "balance_sheet (annual)")
summarize_stmt(t.cashflow, "cashflow (annual)")

In [ ]:
# 季度
summarize_stmt(t.quarterly_income_stmt, "income_stmt (quarterly)")
summarize_stmt(t.quarterly_balance_sheet, "balance_sheet (quarterly)")
summarize_stmt(t.quarterly_cashflow, "cashflow (quarterly)")

## 5) 快速定位：现金流里“CFO / CAPEX”到底叫什么？

不同公司/时期字段名可能略有差异。这里做一个“模糊搜索”，帮你找：  
- Operating Cash Flow（经营现金流）  
- Capital Expenditure（资本开支）  
- Depreciation（折旧摊销）


In [ ]:
def find_line_items(df: pd.DataFrame, patterns):
    if df is None or df.empty:
        return {}
    idx = pd.Index(df.index.astype(str))
    out = {}
    for p in patterns:
        hits = [s for s in idx if p.lower() in s.lower()]
        out[p] = hits[:20]
    return out

patterns = ["operat", "capital", "capex", "deprec", "amort", "ppe", "cash"]
find_line_items(t.cashflow, patterns)

## 6)（可选）导出：把当前 ticker 的结构快照保存为本地 CSV

后续你可以用它做“字段映射表”，决定估值模块最小字段集合。


In [ ]:
import os

out_dir = "yfinance_snapshots"
os.makedirs(out_dir, exist_ok=True)

def safe_to_csv(df, path):
    if df is None or df.empty:
        return False
    df.to_csv(path)
    return True

saved = {}
saved["info"] = os.path.join(out_dir, f"{ticker}_info.csv")
info_df.to_csv(saved["info"], index=False)

saved["income_annual"] = os.path.join(out_dir, f"{ticker}_income_annual.csv")
saved["balance_annual"] = os.path.join(out_dir, f"{ticker}_balance_annual.csv")
saved["cashflow_annual"] = os.path.join(out_dir, f"{ticker}_cashflow_annual.csv")

saved["income_quarterly"] = os.path.join(out_dir, f"{ticker}_income_quarterly.csv")
saved["balance_quarterly"] = os.path.join(out_dir, f"{ticker}_balance_quarterly.csv")
saved["cashflow_quarterly"] = os.path.join(out_dir, f"{ticker}_cashflow_quarterly.csv")

ok = {}
ok["income_annual"] = safe_to_csv(t.income_stmt, saved["income_annual"])
ok["balance_annual"] = safe_to_csv(t.balance_sheet, saved["balance_annual"])
ok["cashflow_annual"] = safe_to_csv(t.cashflow, saved["cashflow_annual"])

ok["income_quarterly"] = safe_to_csv(t.quarterly_income_stmt, saved["income_quarterly"])
ok["balance_quarterly"] = safe_to_csv(t.quarterly_balance_sheet, saved["balance_quarterly"])
ok["cashflow_quarterly"] = safe_to_csv(t.quarterly_cashflow, saved["cashflow_quarterly"])

pd.DataFrame([{"file": k, "path": v, "saved": ok.get(k, True)} for k, v in saved.items()])

## 下一步你要关注什么（建议）

1) `cashflow` 里能否稳定找到：CFO、CAPEX、折旧摊销。  
2) `balance_sheet` 里能否找到：现金、总债务（短债/长债）。  
3) `info` 里股本是否稳定；如果不稳定，是否能用 `marketCap/currentPrice` 反推。

你把某个 ticker 的 `cashflow (annual)` 那一页截图/贴出来，我可以帮你把“字段映射表”定下来，直接为 DCF 做准备。
